# **AdaGrad**

Adagrad (Adaptive Gradient Algorithm) is a landmark optimizer in deep learning. It was the first optimizer to introduce the concept of an adaptive learning rate, meaning it customizes the step size for every single weight in your neural network individually.

Adagrad works like a personalized learning assistant that customizes the study pace for every single weight in a neural network individually. Instead of updating all weights at the same speed, it tracks how active each weight has been during training. For weights that are updated frequently (like common features in your dataset), Adagrad gradually shrinks their learning rate to take small, careful steps so they don't overshoot. For weights that are updated rarely (like sparse or rare features), it keeps their learning rate relatively large, allowing the model to make big, significant adjustments and learn quickly the moment those rare features finally appear.

## 1. The Core Intuition: Rare Words vs. Common Words

Imagine you are learning a new language:
* You see the word "the" (common) in almost every sentence. You don't need to change your brain's language model much when you see it. You take tiny learning steps.
* You see the word "defenestration" (extremely rare). When you finally see it, it carries a lot of new information. You need to take a much larger mental learning step to remember it.

Adagrad does exactly this for weights:
* Frequently updated weights (common features) get their learning rate scaled down, taking smaller, careful steps.
* Infrequently updated weights (rare features) get their learning rate scaled up, taking larger steps so they can learn quickly when they finally appear.

## 2. The Mathematics: How It Adapts

Instead of having one flat learning rate for the entire network, Adagrad keeps a running memory of the history of gradients for each individual weight.

### Step 1: Accumulate the squared gradients
For each weight, Adagrad tracks the sum of the squares of all its past gradients ($S_t$):

$$S_t = S_{t-1} + g_t^2$$

### Step 2: Update the weight
When updating the weight, we divide the learning rate by the square root of this accumulated history ($S_t$):

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{S_t + \epsilon}} \cdot g_t$$

### Symbol Legend:
* **$w_t$:** Current weight value.
* **$g_t$:** The raw gradient at the current step.
* **$S_t$:** The accumulated sum of squared gradients for this weight.
* **$\eta$ (Eta):** The global starting learning rate.
* **$\epsilon$ (Epsilon):** A tiny number (like `1e-8`) to prevent division by zero.

If a weight has had huge gradients in the past, $S_t$ will be very large, making the step size tiny. If a weight has had tiny or rare gradients in the past, $S_t$ will be very small, keeping the step size relatively large.

## 3. The Fatal Flaw: The Freeze

While the math is beautiful, Adagrad has one massive problem: The accumulator $S_t$ only grows.

Because we square the gradients ($g_t^2$), we are continuously adding positive numbers to $S_t$ at every single training step. As training goes on:
1. $S_t$ becomes extremely large.
2. The denominator $\sqrt{S_t}$ becomes huge.
3. The effective learning rate ($\frac{\eta}{\sqrt{S_t}}$) shrinks closer and closer to zero.

Eventually, the model's learning rate completely freezes. The network stops learning entirely, even if it is still far away from the global minimum.

This fatal flaw is exactly why researchers invented RMSprop (which uses EWMA on the squared gradients so the memory fades and doesn't grow to infinity) and Adam.

## 4. Where is Adagrad Used?

Because of its freeze problem, we rarely use Adagrad for standard deep neural networks (like CNNs or MLPs). 

However, it is still incredibly useful for Sparse Data tasks:
* **Natural Language Processing (NLP) & Word Embeddings:** In text, most words are rare, while a few are very common. Adagrad excels at updating the weights of rare words quickly without destroying the weights of common words.

## 5. In Keras Code

```python
from tensorflow import keras

# Define Adagrad optimizer
opt = keras.optimizers.Adagrad(learning_rate=0.01)

# Compile model
model.compile(optimizer=opt, loss='binary_crossentropy')
```

---

# The Mathematics of the Adagrad Optimizer

Adagrad (Adaptive Gradient Algorithm) scales the learning rate of each parameter individually based on the history of its gradients. It does this by dividing the global learning rate by the running sum of squares of all past gradients.

## 1. The Core Equations

At each training step $t$, the optimizer updates every weight $w$ using two mathematical steps:

### Step 1: Accumulate the Squared Gradients

$$S_t = S_{t-1} + g_t^2$$

### Step 2: Update the Weight

$$w_{t+1} = w_t - \frac{\eta}{\sqrt{S_t + \epsilon}} \cdot g_t$$

## 2. Symbol Legend

* **$w_t$:** The weight value at the current step $t$.
* **$w_{t+1}$:** The updated weight value for the next step.
* **$g_t$:** The gradient calculated for the weight at the current step.
* **$S_t$:** The running sum of the squares of all past gradients for this specific weight.
* **$S_{t-1}$:** The running sum of squared gradients from the previous step.
* **$\eta$ (Eta):** The global base learning rate (set by the user).
* **$\epsilon$ (Epsilon):** A tiny constant (e.g., $10^{-8}$) added to prevent division by zero.

## 3. Step-by-Step Mathematical Walkthrough

Let's calculate the updates for a single weight over 3 steps using:
* **Starting Weight ($w_1$):** $0.5$
* **Starting Sum ($S_0$):** $0.0$
* **Learning Rate ($\eta$):** $0.1$
* **Epsilon ($\epsilon$):** Negligible for this simple calculation

### Step 1 (Current Gradient: $g_1 = 2.0$)
Calculate the new running sum ($S_1$):
$$S_1 = S_0 + g_1^2 = 0 + (2.0)^2 = \mathbf{4.0}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{S_1}} = \frac{0.1}{\sqrt{4.0}} = \frac{0.1}{2.0} = \mathbf{0.05}$$

Update the weight ($w_2$):
$$w_2 = w_1 - \eta_{\text{eff}} \cdot g_1 = 0.5 - (0.05 \cdot 2.0) = \mathbf{0.4}$$

### Step 2 (Current Gradient: $g_2 = 1.0$)
Calculate the new running sum ($S_2$):
$$S_2 = S_1 + g_2^2 = 4.0 + (1.0)^2 = \mathbf{5.0}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{S_2}} = \frac{0.1}{\sqrt{5.0}} \approx \frac{0.1}{2.236} \approx \mathbf{0.0447}$$

Update the weight ($w_3$):
$$w_3 = w_2 - \eta_{\text{eff}} \cdot g_2 = 0.4 - (0.0447 \cdot 1.0) = \mathbf{0.3553}$$

### Step 3 (Current Gradient: $g_3 = 0.5$)
Calculate the new running sum ($S_3$):
$$S_3 = S_2 + g_3^2 = 5.0 + (0.5)^2 = \mathbf{5.25}$$

Calculate the effective learning rate:
$$\eta_{\text{eff}} = \frac{\eta}{\sqrt{S_3}} = \frac{0.1}{\sqrt{5.25}} \approx \frac{0.1}{2.291} \approx \mathbf{0.0436}$$

Update the weight ($w_4$):
$$w_4 = w_3 - \eta_{\text{eff}} \cdot g_3 = 0.3553 - (0.0436 \cdot 0.5) \approx \mathbf{0.3335}$$

## 4. Why the Effective Learning Rate Freezes

Notice how the **effective learning rate ($\eta_{\text{eff}}$)** decreases at every step:
$$\text{Step 1: } 0.05 \longrightarrow \text{Step 2: } 0.0447 \longrightarrow \text{Step 3: } 0.0436$$

Because $S_t = S_{t-1} + g_t^2$, we are continuously adding positive squared numbers to $S_t$. 

As the number of training steps $t$ grows to thousands or millions:
$$\lim_{t \to \infty} S_t = \infty$$

Since $S_t$ approaches infinity, the denominator in our update step approaches infinity as well, which forces the weight step size to zero:
$$\lim_{t \to \infty} \frac{\eta}{\sqrt{S_t + \epsilon}} = 0$$

This mathematically forces the weights to stop updating, freezing the model before it can reach the global minimum.